# Download des LUNA16-Datensatzes

Bevor wir mit der Analyse beginnen, benoetigen wir die Daten. Der LUNA16-Datensatz wird auf Zenodo in zwei Teilen gehostet:

- **Part 1** (https://zenodo.org/records/3723295): `annotations.csv`, `candidates.csv` und subset0 bis subset6
- **Part 2** (https://zenodo.org/records/4121926): subset7 bis subset9

Jedes Subset enthaelt CT-Scans im Format `.mhd` + `.raw` (Dateipaare). Der gesamte Download betraegt ~67 GB, seien Sie also vorbereitet.

Der folgende Code unterstuetzt **Resume** (Fortsetzen) bei einem Verbindungsabbruch. Bereits existierende Dateien werden automatisch uebersprungen.

In [ ]:
import os
import zipfile
from pathlib import Path
import shutil

import requests
from tqdm import tqdm

## Konfiguration

Wir definieren die Pfade und URLs fuer jede Datei. Die endgueltige Struktur befindet sich in `data/luna/` mit den CSVs und den Subsets.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LUNA_DIR = PROJECT_ROOT / "data" / "luna"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LUNA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projekt: {PROJECT_ROOT}")
print(f"Downloads: {RAW_DIR}")
print(f"Datensatz: {LUNA_DIR}")

In [ ]:
ZENODO_PART1 = "https://zenodo.org/records/3723295/files"
ZENODO_PART2 = "https://zenodo.org/records/4121926/files"

LUNA16_FILES = {
    "annotations.csv": f"{ZENODO_PART1}/annotations.csv?download=1",
    "candidates.csv": f"{ZENODO_PART1}/candidates.csv?download=1",
    "subset0.zip": f"{ZENODO_PART1}/subset0.zip?download=1",
    "subset1.zip": f"{ZENODO_PART1}/subset1.zip?download=1",
    "subset2.zip": f"{ZENODO_PART1}/subset2.zip?download=1",
    "subset3.zip": f"{ZENODO_PART1}/subset3.zip?download=1",
    "subset4.zip": f"{ZENODO_PART1}/subset4.zip?download=1",
    "subset5.zip": f"{ZENODO_PART1}/subset5.zip?download=1",
    "subset6.zip": f"{ZENODO_PART1}/subset6.zip?download=1",
    "subset7.zip": f"{ZENODO_PART2}/subset7.zip?download=1",
    "subset8.zip": f"{ZENODO_PART2}/subset8.zip?download=1",
    "subset9.zip": f"{ZENODO_PART2}/subset9.zip?download=1",
}

print(f"Gesamtzahl der Dateien: {len(LUNA16_FILES)}")

## Download-Funktionen

Die folgenden Funktionen uebernehmen das Herunterladen und Entpacken der Dateien. Der Download unterstuetzt das Fortsetzen (Resume), wenn die Verbindung abbricht, und ueberspringt Dateien, die bereits existieren.

In [ ]:
def download_file(url, dest_path, chunk_size=8192):
    """Laedt eine Datei mit Resume-Unterstuetzung herunter."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    head = requests.head(url, allow_redirects=True, timeout=30)
    remote_size = int(head.headers.get("Content-Length", 0))

    if dest_path.exists():
        local_size = dest_path.stat().st_size
        if local_size >= remote_size and remote_size > 0:
            print(f"  [OK] {dest_path.name} existiert bereits ({local_size / 1e6:.1f} MB)")
            return False
    else:
        local_size = 0

    headers = {}
    if local_size > 0:
        headers["Range"] = f"bytes={local_size}-"
        print(f"  Fortsetzen von {dest_path.name} ab {local_size / 1e6:.1f} MB...")

    response = requests.get(url, headers=headers, stream=True, timeout=60)
    response.raise_for_status()

    content_length = int(response.headers.get("Content-Length", 0))
    total = content_length if content_length > 0 else (remote_size - local_size)
    mode = "ab" if local_size > 0 else "wb"

    with open(dest_path, mode) as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc=dest_path.name) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return True

In [ ]:
def extract_subset_zip(zip_path, luna_dir):
    """Entpackt ein ZIP-Subset in die korrekte Struktur."""
    zip_path = Path(zip_path)
    luna_dir = Path(luna_dir)
    subset_name = zip_path.stem
    target_dir = luna_dir / subset_name

    if target_dir.exists() and any(target_dir.glob("*.mhd")):
        n = len(list(target_dir.glob("*.mhd")))
        print(f"  [OK] {subset_name} bereits entpackt ({n} scans)")
        return

    print(f"  Entpacken von {zip_path.name}...")
    temp_dir = luna_dir / f"_temp_{subset_name}"

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_dir)

    # ZIP erstellt verschachtelten Ordner (subset0/subset0/) -- auf die richtige Ebene verschieben
    nested_dir = temp_dir / subset_name / subset_name
    if nested_dir.exists():
        source = nested_dir
    elif (temp_dir / subset_name).exists():
        source = temp_dir / subset_name
    else:
        source = temp_dir

    target_dir.mkdir(parents=True, exist_ok=True)
    for f in source.iterdir():
        shutil.move(str(f), str(target_dir / f.name))

    shutil.rmtree(temp_dir, ignore_errors=True)
    mhd_count = len(list(target_dir.glob("*.mhd")))
    print(f"  [OK] {subset_name} entpackt ({mhd_count} scans)")

In [ ]:
def download_and_extract(file_name, url, raw_dir, luna_dir):
    """Laedt eine Datei aus LUNA16 herunter und entpackt sie."""
    raw_dir, luna_dir = Path(raw_dir), Path(luna_dir)

    if file_name.endswith(".csv"):
        dest = luna_dir / file_name
        if dest.exists() and dest.stat().st_size > 0:
            print(f"  [OK] {file_name} existiert bereits unter luna/")
            return
        download_file(url, dest)
        return

    if file_name.endswith(".zip"):
        subset_name = file_name.replace(".zip", "")
        target_dir = luna_dir / subset_name
        if target_dir.exists() and any(target_dir.glob("*.mhd")):
            print(f"  [OK] {subset_name} existiert bereits -- Download wird uebersprungen")
            return
        zip_path = raw_dir / file_name
        download_file(url, zip_path)
        extract_subset_zip(zip_path, luna_dir)

## Test mit einem Subset

Bevor wir den gesamten Datensatz (~67 GB) herunterladen, testen wir mit einem einzigen Subset, um sicherzustellen, dass alles funktioniert:

In [ ]:
test_files = ["annotations.csv", "candidates.csv", "subset0.zip"]

for file_name in test_files:
    print(f"\nVerarbeitung: {file_name}")
    url = LUNA16_FILES[file_name]
    download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)

## Schnelle Ueberpruefung

Sehen wir uns an, was wir bisher in `data/luna/` haben:

In [ ]:
for item in sorted(LUNA_DIR.iterdir()):
    if item.name.startswith("."):
        continue
    if item.is_dir():
        mhd_files = list(item.glob("*.mhd"))
        print(f"  {item.name}/  ({len(mhd_files)} scans)")
    else:
        print(f"  {item.name}  ({item.stat().st_size / 1e6:.1f} MB)")

## Vollstaendiger Download

Wenn der obige Test funktioniert hat, fuehren Sie die folgende Zelle aus, um alle 10 Subsets herunterzuladen. Bereits vorhandene Dateien werden automatisch uebersprungen.

In [ ]:
for file_name, url in LUNA16_FILES.items():
    print(f"\nVerarbeitung: {file_name}")
    try:
        download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)
    except Exception as e:
        print(f"  [FEHLER] {file_name}: {e}")
        print(f"  Fuehren Sie diese Zelle erneut aus, um es noch einmal zu versuchen.")

## Abschliessende Ueberpruefung

In [ ]:
for csv_name in ["annotations.csv", "candidates.csv"]:
    csv_path = LUNA_DIR / csv_name
    status = "vorhanden" if csv_path.exists() else "FEHLT"
    print(f"  {csv_name}: {status}")

print()
total_scans = 0
for i in range(10):
    subset_dir = LUNA_DIR / f"subset{i}"
    if subset_dir.exists():
        n = len(list(subset_dir.glob("*.mhd")))
        total_scans += n
        print(f"  subset{i}: {n} scans")
    else:
        print(f"  subset{i}: FEHLT")

print(f"\nGesamtzahl der CT-Scans: {total_scans}")